# Intelligent Stock Market Trend Predictor using ML
Step-by-step notebook for viva. *Educational only - not financial advice.*

## Step 1 - Data Collection (yfinance)
We download all available history; missing values are forward-filled.

In [ ]:
import pandas as pd, numpy as np, yfinance as yf, matplotlib.pyplot as plt
from features import add_features, time_split, FEATURES, SEQ_LEN
df = yf.download('TCS.NS', period='max', auto_adjust=True, progress=False)
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
df = df.dropna(how='all').ffill()
print(df.shape); df.tail()

## Step 2 - Feature Engineering
MA20, MA50, RSI(14), MACD, Bollinger Bands (ta library) + lag returns. Target = 1 if Close(t+1) > Close(t).

In [ ]:
d = add_features(df).dropna(subset=FEATURES+['Target'])
d[['Close','MA20','MA50','RSI','MACD','Target']].tail()

In [ ]:
plt.figure(figsize=(11,4)); d['Close'].plot(label='Close'); d['MA20'].plot(label='MA20'); d['MA50'].plot(label='MA50'); plt.legend(); plt.show()

## Step 3 - Time-based split (80-20, NO shuffle)
**Data leakage:** shuffling mixes future rows into training, so the model would 'know' the future and show fake accuracy.

In [ ]:
train, test = time_split(d, 0.8)
print('Train:', train.index[0].date(), '->', train.index[-1].date()); print('Test :', test.index[0].date(), '->', test.index[-1].date())

## Step 4 - Model A: RandomForest (baseline)
Simple, explainable, resists overfitting; any advanced model must beat it.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=42).fit(train[FEATURES], train['Target'])
rf_pred = rf.predict(test[FEATURES])
print('Accuracy', accuracy_score(test['Target'], rf_pred)); print('Naive baseline', max(test['Target'].mean(), 1-test['Target'].mean()))
print(confusion_matrix(test['Target'], rf_pred))
pd.Series(rf.feature_importances_, FEATURES).sort_values().plot.barh(); plt.show()

## Step 5 - Model B: LSTM
Gates: **forget** (drop old info), **input** (store new info), **output** (what to pass on); the **cell state** carries long-term memory. Input = last 60 days; 2 LSTM layers (50 units), Dropout 0.2, Dense(1, sigmoid), Adam.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(train[FEATURES])          # fit on TRAIN only (no leakage)
def seqs(part):
    v = scaler.transform(part[FEATURES]); t = part['Target'].values
    X = np.array([v[i-SEQ_LEN:i] for i in range(SEQ_LEN, len(part))], dtype='float32'); y = t[SEQ_LEN-1:len(part)-1]
    return X, y
Xtr, ytr = seqs(train); Xte, yte = seqs(test); Xtr.shape

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
m = Sequential([LSTM(50, return_sequences=True, input_shape=(SEQ_LEN, len(FEATURES))), Dropout(0.2), LSTM(50), Dropout(0.2), Dense(1, activation='sigmoid')])
m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
h = m.fit(Xtr, ytr, epochs=30, batch_size=32, validation_split=0.1, shuffle=False, callbacks=[EarlyStopping(patience=4, restore_best_weights=True)], verbose=0)
lstm_pred = (m.predict(Xte, verbose=0).ravel() > 0.5).astype(int)
print('LSTM accuracy', accuracy_score(yte, lstm_pred)); print(confusion_matrix(yte, lstm_pred))

## Step 6 - Compare models

In [ ]:
res = pd.DataFrame({'RandomForest':[accuracy_score(test['Target'],rf_pred),precision_score(test['Target'],rf_pred,zero_division=0),recall_score(test['Target'],rf_pred,zero_division=0),f1_score(test['Target'],rf_pred,zero_division=0)],
 'LSTM':[accuracy_score(yte,lstm_pred),precision_score(yte,lstm_pred,zero_division=0),recall_score(yte,lstm_pred,zero_division=0),f1_score(yte,lstm_pred,zero_division=0)]}, index=['Accuracy','Precision','Recall','F1'])
res.plot.bar(figsize=(7,4)); plt.show(); res

## Step 7 - Save models (.pkl and .h5)

In [ ]:
import joblib; joblib.dump(rf, 'rf_single_model.pkl'); m.save('lstm_single_model.h5')

## Conclusion
Trend prediction accuracy near 50-56% is realistic. Always compare with the naive baseline. Future scope: news sentiment analysis.